# BioFuse Tutorial 4: Adding a New Classifier

This tutorial shows you how to add custom classifiers to BioFuse.

We'll cover:
1. Understanding BioFuse's classifier architecture
2. Creating a simple custom classifier
3. Implementing a neural network classifier
4. Registering and using your classifier
5. Real-world examples

## Why Custom Classifiers?

While BioFuse includes common classifiers (Logistic Regression, XGBoost, CatBoost, Neural Nets), you might want to add:
- Specialized algorithms (e.g., SVM with custom kernels)
- Domain-specific models
- Ensemble methods
- Research prototypes

## Part 1: Understanding the Classifier Architecture

All BioFuse classifiers inherit from `BaseClassifier` and must implement three key methods:

1. **`fit(X, y, X_val, y_val)`**: Train the classifier
2. **`predict(X)`**: Predict class labels
3. **`predict_proba(X)`**: Predict class probabilities

### The BaseClassifier Interface

In [ ]:
from abc import ABC, abstractmethod
import numpy as np
from typing import Optional

class BaseClassifier(ABC):
    """Abstract base class for all classifiers."""
    
    def __init__(self, **kwargs):
        self.is_fitted = False
        self.num_classes = None
        self.multi_label = False
    
    @abstractmethod
    def fit(self, X: np.ndarray, y: np.ndarray, 
            X_val: Optional[np.ndarray] = None,
            y_val: Optional[np.ndarray] = None) -> 'BaseClassifier':
        """Train the classifier."""
        pass
    
    @abstractmethod
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels."""
        pass
    
    @abstractmethod
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities."""
        pass

print("BaseClassifier interface defined")

## Part 2: Creating a Simple Classifier - SVM Example

Let's create a Support Vector Machine classifier.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
import numpy as np
from typing import Optional

# Import BioFuse base class
from biofuse.classifiers import BaseClassifier, ClassifierFactory

class SVMClassifier(BaseClassifier):
    """
    Support Vector Machine classifier for BioFuse.
    
    Uses sklearn's SVC with probability calibration.
    """
    
    def __init__(
        self,
        kernel: str = 'rbf',
        C: float = 1.0,
        gamma: str = 'scale',
        random_state: int = 42,
        **kwargs
    ):
        """
        Initialize SVM classifier.
        
        Args:
            kernel: Kernel type ('linear', 'rbf', 'poly', 'sigmoid')
            C: Regularization parameter
            gamma: Kernel coefficient
            random_state: Random seed
            **kwargs: Additional SVC parameters
        """
        super().__init__()
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.random_state = random_state
        self.kwargs = kwargs
        
        self.scaler = StandardScaler()
        self.classifier = None
    
    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        X_val: Optional[np.ndarray] = None,
        y_val: Optional[np.ndarray] = None
    ) -> 'SVMClassifier':
        """Train the SVM classifier."""
        # Scale features (important for SVM!)
        X_scaled = self.scaler.fit_transform(X)
        
        # Determine number of classes
        self.num_classes = len(np.unique(y))
        self.multi_label = len(y.shape) > 1 and y.shape[1] > 1
        
        # Create SVM
        base_svm = SVC(
            kernel=self.kernel,
            C=self.C,
            gamma=self.gamma,
            random_state=self.random_state,
            probability=False,  # Faster without probability
            **self.kwargs
        )
        
        # Wrap with calibration for probability estimates
        # This allows us to get predict_proba
        self.classifier = CalibratedClassifierCV(
            base_svm,
            cv=3,
            method='sigmoid'
        )
        
        # Train
        print(f"Training SVM with {self.kernel} kernel...")
        self.classifier.fit(X_scaled, y)
        self.is_fitted = True
        
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels."""
        if not self.is_fitted:
            raise RuntimeError("Classifier must be fitted before prediction")
        
        X_scaled = self.scaler.transform(X)
        return self.classifier.predict(X_scaled)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities."""
        if not self.is_fitted:
            raise RuntimeError("Classifier must be fitted before prediction")
        
        X_scaled = self.scaler.transform(X)
        return self.classifier.predict_proba(X_scaled)
    
    def get_params(self):
        """Get classifier parameters."""
        return {
            'kernel': self.kernel,
            'C': self.C,
            'gamma': self.gamma,
            'random_state': self.random_state,
            **self.kwargs
        }

print("SVMClassifier created!")

### Step 2: Register the classifier

In [ ]:
# Register the classifier with the factory
ClassifierFactory.register('svm', SVMClassifier)
ClassifierFactory.register('svc', SVMClassifier)  # Alternative name

# Check it's registered
print("Available classifiers:", ClassifierFactory.list_available())

### Step 3: Test the classifier

In [ ]:
# Test on a small dataset
from biofuse import BioFuse, load_medmnist, get_classifier, compute_metrics

# Load small subset for testing
train_data, num_classes = load_medmnist('pathmnist', split='train')
test_data, _ = load_medmnist('pathmnist', split='test')

# Extract embeddings (cached if already computed)
biofuse = BioFuse(models=['BioMedCLIP'], fusion_method='concat')

train_emb, train_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='train'
)

test_emb, test_labels, _, _, _ = biofuse.generate_embeddings(
    train_data=None,
    dataset_type='medmnist',
    dataset_name='pathmnist',
    split='test'
)

# Use subset for faster testing
train_emb_small = train_emb[:1000]
train_labels_small = train_labels[:1000]

# Create and train SVM classifier
svm_clf = get_classifier('svm', kernel='rbf', C=1.0)
svm_clf.fit(train_emb_small, train_labels_small)

# Evaluate
pred = svm_clf.predict(test_emb)
proba = svm_clf.predict_proba(test_emb)

metrics = compute_metrics(test_labels, pred, proba, num_classes, task='multi-class')
print(f"\nSVM Accuracy: {metrics['accuracy']:.4f}")
print(f"SVM AUC: {metrics['auc_macro']:.4f}")

## Part 3: Adding to BioFuse Codebase

To make your classifier permanently available:

### Step 1: Create file `biofuse/classifiers/svm_classifier.py`

In [ ]:
# Content for biofuse/classifiers/svm_classifier.py

"""
\"\"\"Support Vector Machine classifier for BioFuse.\"\"\"

from typing import Optional, Dict, Any
import numpy as np
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV

from .base import BaseClassifier, ClassifierFactory


class SVMClassifier(BaseClassifier):
    # ... (same implementation as above)
    pass


# Register the classifier
ClassifierFactory.register('svm', SVMClassifier)
ClassifierFactory.register('svc', SVMClassifier)
"""

print("Create this file: biofuse/classifiers/svm_classifier.py")

### Step 2: Update `biofuse/classifiers/__init__.py`

In [ ]:
# Add to biofuse/classifiers/__init__.py

"""
from .base import BaseClassifier, ClassifierFactory, get_classifier
from .sklearn_classifiers import (
    LogisticRegression,
    XGBoostClassifier,
    CatBoostClassifierWrapper
)
from .neural import NeuralNetClassifier
from .svm_classifier import SVMClassifier  # <-- Add this

__all__ = [
    'BaseClassifier',
    'ClassifierFactory',
    'get_classifier',
    'LogisticRegression',
    'XGBoostClassifier',
    'CatBoostClassifierWrapper',
    'NeuralNetClassifier',
    'SVMClassifier',  # <-- Add this
]
"""

print("Update biofuse/classifiers/__init__.py")

### Step 3: Use it everywhere!

In [ ]:
# Python API
from biofuse import get_classifier
clf = get_classifier('svm', kernel='linear', C=0.5)

# CLI
!biofuse train --dataset pathmnist --models BioMedCLIP --classifier svm

# Config file
"""
name: svm_experiment
data:
  dataset: pathmnist
model:
  models: [BioMedCLIP]
classifier:
  type: svm
  kernel: rbf
  C: 1.0
"""

## Part 4: Advanced Example - Ensemble Classifier

Let's create a more sophisticated classifier that combines multiple models.

In [ ]:
from biofuse.classifiers import BaseClassifier, ClassifierFactory, get_classifier
import numpy as np
from typing import List, Optional

class VotingEnsembleClassifier(BaseClassifier):
    """
    Ensemble classifier that combines predictions from multiple classifiers.
    
    Supports both hard voting (majority vote) and soft voting (average probabilities).
    """
    
    def __init__(
        self,
        classifier_names: List[str] = None,
        classifier_params: List[dict] = None,
        voting: str = 'soft',
        weights: List[float] = None,
        **kwargs
    ):
        """
        Initialize ensemble classifier.
        
        Args:
            classifier_names: List of classifier names to ensemble
            classifier_params: List of parameter dicts for each classifier
            voting: 'hard' or 'soft'
            weights: Optional weights for each classifier
        """
        super().__init__()
        
        # Default: logistic, xgboost, catboost
        if classifier_names is None:
            classifier_names = ['logistic', 'xgboost', 'catboost']
        
        if classifier_params is None:
            classifier_params = [{} for _ in classifier_names]
        
        self.classifier_names = classifier_names
        self.classifier_params = classifier_params
        self.voting = voting
        self.weights = weights or [1.0] * len(classifier_names)
        
        self.classifiers = []
    
    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        X_val: Optional[np.ndarray] = None,
        y_val: Optional[np.ndarray] = None
    ) -> 'VotingEnsembleClassifier':
        """Train all classifiers in the ensemble."""
        self.num_classes = len(np.unique(y))
        self.classifiers = []
        
        print(f"Training ensemble of {len(self.classifier_names)} classifiers...")
        
        for name, params in zip(self.classifier_names, self.classifier_params):
            print(f"  Training {name}...")
            clf = get_classifier(name, num_classes=self.num_classes, **params)
            clf.fit(X, y, X_val, y_val)
            self.classifiers.append(clf)
        
        self.is_fitted = True
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict using ensemble voting."""
        if not self.is_fitted:
            raise RuntimeError("Ensemble must be fitted before prediction")
        
        if self.voting == 'hard':
            # Majority vote
            predictions = np.array([clf.predict(X) for clf in self.classifiers])
            # Weighted voting
            weighted_votes = predictions.T @ self.weights
            return np.argmax(weighted_votes, axis=1)
        else:
            # Soft voting: average probabilities
            proba = self.predict_proba(X)
            return np.argmax(proba, axis=1)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict probabilities using weighted average."""
        if not self.is_fitted:
            raise RuntimeError("Ensemble must be fitted before prediction")
        
        # Get probabilities from all classifiers
        all_probs = np.array([clf.predict_proba(X) for clf in self.classifiers])
        
        # Weighted average
        weights = np.array(self.weights) / np.sum(self.weights)
        weighted_probs = np.tensordot(weights, all_probs, axes=([0], [0]))
        
        return weighted_probs


# Register
ClassifierFactory.register('ensemble', VotingEnsembleClassifier)
ClassifierFactory.register('voting', VotingEnsembleClassifier)

print("VotingEnsembleClassifier created!")

### Test the ensemble

In [ ]:
# Create ensemble classifier
ensemble = get_classifier(
    'ensemble',
    classifier_names=['logistic', 'xgboost'],
    voting='soft',
    weights=[0.4, 0.6]  # Give XGBoost more weight
)

# Train on small subset (for speed)
ensemble.fit(train_emb_small, train_labels_small)

# Predict
pred = ensemble.predict(test_emb)
proba = ensemble.predict_proba(test_emb)

# Evaluate
metrics = compute_metrics(test_labels, pred, proba, num_classes, task='multi-class')
print(f"\nEnsemble Accuracy: {metrics['accuracy']:.4f}")
print(f"Ensemble AUC: {metrics['auc_macro']:.4f}")

## Part 5: Neural Network Classifier Template

For PyTorch-based classifiers, here's a template:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from biofuse.classifiers import BaseClassifier, ClassifierFactory
import numpy as np

class SimpleNNClassifier(BaseClassifier):
    """
    Simple feedforward neural network classifier.
    """
    
    def __init__(
        self,
        hidden_dims: list = None,
        learning_rate: float = 0.001,
        epochs: int = 50,
        batch_size: int = 32,
        dropout: float = 0.3,
        device: str = 'cuda' if torch.cuda.is_available() else 'cpu',
        **kwargs
    ):
        super().__init__()
        self.hidden_dims = hidden_dims or [256, 128]
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.dropout = dropout
        self.device = device
        
        self.model = None
    
    def _build_model(self, input_dim: int, num_classes: int):
        """Build the neural network."""
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in self.hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(self.dropout)
            ])
            prev_dim = hidden_dim
        
        # Output layer
        layers.append(nn.Linear(prev_dim, num_classes))
        
        return nn.Sequential(*layers).to(self.device)
    
    def fit(
        self,
        X: np.ndarray,
        y: np.ndarray,
        X_val: Optional[np.ndarray] = None,
        y_val: Optional[np.ndarray] = None
    ) -> 'SimpleNNClassifier':
        """Train the neural network."""
        self.num_classes = len(np.unique(y))
        input_dim = X.shape[1]
        
        # Build model
        self.model = self._build_model(input_dim, self.num_classes)
        
        # Create data loaders
        train_dataset = TensorDataset(
            torch.FloatTensor(X),
            torch.LongTensor(y)
        )
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True
        )
        
        # Setup training
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        
        # Train
        print(f"Training neural network for {self.epochs} epochs...")
        self.model.train()
        
        for epoch in range(self.epochs):
            total_loss = 0
            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(self.device)
                batch_y = batch_y.to(self.device)
                
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            if (epoch + 1) % 10 == 0:
                avg_loss = total_loss / len(train_loader)
                print(f"  Epoch {epoch+1}/{self.epochs}, Loss: {avg_loss:.4f}")
        
        self.is_fitted = True
        return self
    
    def predict(self, X: np.ndarray) -> np.ndarray:
        """Predict class labels."""
        proba = self.predict_proba(X)
        return np.argmax(proba, axis=1)
    
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities."""
        if not self.is_fitted:
            raise RuntimeError("Model must be fitted before prediction")
        
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            logits = self.model(X_tensor)
            proba = torch.softmax(logits, dim=1)
            return proba.cpu().numpy()


# Register
ClassifierFactory.register('simple_nn', SimpleNNClassifier)

print("SimpleNNClassifier created!")

## Summary

### Key Steps to Add a Classifier

1. **Inherit from BaseClassifier**
2. **Implement three methods**:
   - `fit(X, y, X_val, y_val)`
   - `predict(X)`
   - `predict_proba(X)`
3. **Register with ClassifierFactory**
4. **Test thoroughly**
5. **Add to codebase** (optional)

### Checklist

- [ ] Create classifier class inheriting from `BaseClassifier`
- [ ] Implement `fit()` method
- [ ] Implement `predict()` method
- [ ] Implement `predict_proba()` method
- [ ] Set `self.num_classes` in fit
- [ ] Set `self.is_fitted = True` after training
- [ ] Register with `ClassifierFactory.register()`
- [ ] Test on small dataset
- [ ] Verify output shapes
- [ ] Add to `biofuse/classifiers/` (optional)
- [ ] Update `__init__.py` (optional)
- [ ] Create example configs
- [ ] Document parameters and usage

### Best Practices

1. **Feature Scaling**: Many algorithms need scaled features (SVM, Neural Nets)
2. **Probability Calibration**: Ensure `predict_proba` returns well-calibrated probabilities
3. **Validation Support**: Use `X_val` and `y_val` for early stopping
4. **Memory Efficiency**: Clean up after training
5. **Error Handling**: Check `is_fitted` before prediction
6. **Parameter Validation**: Validate inputs in `__init__`

## Example Usage in Production

In [ ]:
# Config file for custom classifier
"""
name: custom_svm_experiment
data:
  dataset: pathmnist
  img_size: 224
model:
  models: [BioMedCLIP, CONCH]
  fusion_method: concat
classifier:
  type: svm
  kernel: rbf
  C: 2.0
  gamma: scale
"""

# Run with CLI
!biofuse train --config custom_svm_experiment.yaml

## Congratulations!

You now know how to:

1. ✅ Create custom classifiers for BioFuse
2. ✅ Implement sklearn-based classifiers
3. ✅ Implement PyTorch-based classifiers
4. ✅ Build ensemble classifiers
5. ✅ Register and use classifiers in CLI and configs

## Resources

- [BioFuse Classifier API](../../biofuse/classifiers/)
- [sklearn Documentation](https://scikit-learn.org/)
- [PyTorch Documentation](https://pytorch.org/docs/)
- [Example Configs](../configs/)